<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day15/Simple_Attention_mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task : Self Attention, Word Embedding

In [1]:
import torch


# input sequence / sentence:
#  "Can you help me to translate this sentence"

sentence = torch.tensor(
    [0, # can
     7, # you
     1, # help
     2, # me
     5, # to
     6, # translate
     4, # this
     3] # sentence
)

sentence

tensor([0, 7, 1, 2, 5, 6, 4, 3])

# Convert word encoded values to embeddings

embeddings size = 16 ( per word )

In [13]:
torch.manual_seed(7)
embed = torch.nn.Embedding(10, 100) # 16 -> Embedding Length, 10-> Vocabulary Size
embedded_sentence = embed(sentence).detach()
embedded_sentence.shape

torch.Size([8, 100])

# Calculate Context weights of every word in sentence

- Simple self attention : Context weight of one word is dependent on all other words of the sentence, words before and words after

- marked self attention : model will only be able to see words before the current and will not see words after the current word

- Scaled dot-product attention : it incolves query, key and values matrix while calculating context weights of a word. here model can process all words from a senetence, words before and after.

In [14]:
omega = torch.empty(8, 8)

for i, x_i in enumerate(embedded_sentence):
    for j, x_j in enumerate(embedded_sentence):
        omega[i, j] = torch.dot(x_i, x_j)

In [15]:
import pandas as pd
pd.DataFrame(omega,columns=["can","you","help","me","to","translate","this","sentence"],
             index=["can","you","help","me","to","translate","this","sentence"])

,can,you,help,me,to,translate,this,sentence
can,96.581413,-0.020498,-2.778799,3.553832,3.626292,0.412726,7.142664,-11.909356
you,-0.020498,111.606346,4.250903,1.658274,3.670578,18.800484,-6.218071,1.088176
help,-2.778799,4.250903,76.436394,8.763985,-13.096233,3.441396,-3.345829,7.002371
me,3.553832,1.658274,8.763985,87.097183,-12.391710,-12.101376,20.859968,0.426350
to,3.626292,3.670578,-13.096233,-12.391710,105.626663,17.897554,10.804946,-0.747243
translate,0.412726,18.800484,3.441396,-12.101376,17.897554,107.711487,1.471389,12.588787
this,7.142664,-6.218071,-3.345829,20.859968,10.804946,1.471389,105.972092,-10.930220
sentence,-11.909356,1.088176,7.002371,0.426350,-0.747243,12.588787,-10.930220,83.138611


In [24]:
import torch.nn.functional as F
import numpy as np
attention_weights = F.softmax(omega, dim=1)/np.sqrt(100)
attention_weights.shape

torch.Size([8, 8])

In [25]:
pd.DataFrame(attention_weights,columns=["can","you","help","me","to","translate","this","sentence"],
             index=["can","you","help","me","to","translate","this","sentence"])

,can,you,help,me,to,translate,this,sentence
can,1.000000e-01,1.107026e-43,7.006492e-45,3.968477e-42,4.266954e-42,1.709584e-43,1.436303e-40,0.000000e+00
you,0.000000e+00,1.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,4.953590e-42,0.000000e+00,0.000000e+00
help,3.956198e-36,4.469290e-33,1.000000e-01,4.076113e-31,1.307608e-40,1.989191e-33,2.243981e-36,7.001448e-32
me,5.218960e-38,7.840653e-39,9.557056e-36,1.000000e-01,5.605194e-45,8.407791e-45,1.712158e-30,2.287355e-39
to,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e-01,7.938370e-40,6.600116e-43,0.000000e+00
translate,0.000000e+00,2.434686e-40,0.000000e+00,0.000000e+00,9.869625e-41,1.000000e-01,0.000000e+00,4.876519e-43
this,1.261169e-44,0.000000e+00,0.000000e+00,1.087114e-38,4.666324e-43,0.000000e+00,1.000000e-01,0.000000e+00
sentence,5.268882e-43,2.322456e-37,8.599079e-35,1.198174e-37,3.705402e-38,2.294048e-32,1.401298e-42,1.000000e-01


In [26]:
# matmul -> Multiplication & Summation both tasks
context_vectors = torch.matmul(
        attention_weights, embedded_sentence)

In [27]:
context_vectors.shape # for every word generate an context based embedding

torch.Size([8, 100])

In [28]:
embedded_sentence[0] # original word embedding for word "can"

tensor([-8.2013e-01,  3.9563e-01,  8.9891e-01, -1.3884e+00, -1.6700e-01,
         2.8515e-01, -6.4109e-01, -8.9366e-01,  9.2654e-01, -5.3551e-01,
        -1.1597e+00, -4.6016e-01,  7.0854e-01,  1.0128e+00,  2.3040e-01,
         1.0902e+00, -1.5827e+00, -3.2457e-01,  1.9264e+00, -3.3001e-01,
         1.9845e-01,  7.8207e-01,  1.0391e+00, -7.2451e-01, -2.0934e-01,
        -2.1534e-01, -1.8157e+00, -3.4524e-01, -2.0615e+00,  6.7410e-01,
        -1.3233e+00, -1.3598e+00, -8.3528e-02, -2.3478e-02,  1.7438e-01,
         2.2983e+00,  9.5710e-01, -6.6187e-01, -8.2845e-01, -6.0568e-01,
        -1.4013e+00,  1.2973e+00,  1.6409e+00, -1.0567e+00, -2.6159e-01,
        -2.5013e-01,  5.0112e-01,  2.6004e-01, -1.7819e-01, -2.5950e-01,
        -1.4488e-02, -3.8389e-01, -2.9662e+00, -1.0606e+00, -3.0900e-01,
         9.3429e-01,  1.6243e+00,  1.5673e-03, -4.3754e-01, -2.1085e+00,
         1.1450e+00, -3.8218e-01, -3.5527e-01,  7.5419e-01,  1.3324e-01,
         1.8255e-01, -5.1463e-01,  8.0052e-01, -1.2

In [29]:
context_vectors[0] # new context based embedding for word "can"

tensor([-8.2013e-02,  3.9563e-02,  8.9891e-02, -1.3884e-01, -1.6700e-02,
         2.8515e-02, -6.4109e-02, -8.9366e-02,  9.2654e-02, -5.3551e-02,
        -1.1597e-01, -4.6016e-02,  7.0854e-02,  1.0128e-01,  2.3040e-02,
         1.0902e-01, -1.5827e-01, -3.2457e-02,  1.9264e-01, -3.3001e-02,
         1.9845e-02,  7.8207e-02,  1.0391e-01, -7.2451e-02, -2.0934e-02,
        -2.1534e-02, -1.8157e-01, -3.4524e-02, -2.0615e-01,  6.7410e-02,
        -1.3233e-01, -1.3598e-01, -8.3528e-03, -2.3478e-03,  1.7438e-02,
         2.2983e-01,  9.5710e-02, -6.6187e-02, -8.2845e-02, -6.0568e-02,
        -1.4013e-01,  1.2973e-01,  1.6409e-01, -1.0567e-01, -2.6159e-02,
        -2.5013e-02,  5.0112e-02,  2.6004e-02, -1.7819e-02, -2.5950e-02,
        -1.4488e-03, -3.8389e-02, -2.9662e-01, -1.0606e-01, -3.0900e-02,
         9.3429e-02,  1.6243e-01,  1.5673e-04, -4.3754e-02, -2.1085e-01,
         1.1450e-01, -3.8218e-02, -3.5527e-02,  7.5419e-02,  1.3324e-02,
         1.8255e-02, -5.1463e-02,  8.0052e-02, -1.2

In [30]:
embedded_sentence[0] - context_vectors[0]

tensor([-7.3812e-01,  3.5607e-01,  8.0902e-01, -1.2496e+00, -1.5030e-01,
         2.5663e-01, -5.7698e-01, -8.0429e-01,  8.3389e-01, -4.8196e-01,
        -1.0437e+00, -4.1414e-01,  6.3769e-01,  9.1148e-01,  2.0736e-01,
         9.8115e-01, -1.4244e+00, -2.9211e-01,  1.7337e+00, -2.9701e-01,
         1.7860e-01,  7.0387e-01,  9.3520e-01, -6.5206e-01, -1.8841e-01,
        -1.9381e-01, -1.6342e+00, -3.1072e-01, -1.8553e+00,  6.0669e-01,
        -1.1910e+00, -1.2238e+00, -7.5175e-02, -2.1130e-02,  1.5694e-01,
         2.0685e+00,  8.6139e-01, -5.9568e-01, -7.4561e-01, -5.4511e-01,
        -1.2611e+00,  1.1676e+00,  1.4768e+00, -9.5102e-01, -2.3543e-01,
        -2.2512e-01,  4.5101e-01,  2.3403e-01, -1.6037e-01, -2.3355e-01,
        -1.3039e-02, -3.4550e-01, -2.6696e+00, -9.5450e-01, -2.7810e-01,
         8.4086e-01,  1.4619e+00,  1.4105e-03, -3.9379e-01, -1.8977e+00,
         1.0305e+00, -3.4397e-01, -3.1974e-01,  6.7877e-01,  1.1992e-01,
         1.6429e-01, -4.6317e-01,  7.2047e-01, -1.1